In [1]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import shutil

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [7]:
coco_path = r'../Data/Raw/DENTEX CHALLENGE 2023/Training_Data\quadrant-enumeration-disease\train_quadrant_enumeration_disease.json'
images_path = r'../Data/Raw/DENTEX CHALLENGE 2023/Training_Data/quadrant-enumeration-disease/xrays'

with open(coco_path) as f:
    coco_file = json.load(f)

In [8]:
# Stage 1
s1_train_path = r'..\Data\Processed\Stage 1 (Disease Detection)\train'
s1_val_path = r'..\Data\Processed\Stage 1 (Disease Detection)\val'
s1_test_path = r'..\Data\Processed\Stage 1 (Disease Detection)\test'

# Stage 2
s2_train_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\train'
s2_val_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\val'
s2_test_path = r'..\Data\Processed\Stage 1 (Disease Classifier)\test'

In [ ]:
diagnosis_map = {
    0: "impacted",
    1: "caries",
    2: "periapical",
    3: "deep_caries"
}


def create_dentex_df(diseases_dict):

    f_name = []
    bbox = []
    img_h = []
    img_w = []
    disease = []
    for item in coco_file['images']:
        for ann in coco_file['annotations']:
            if int(ann['image_id']) == int(item['id']):
                f_name.append(item['file_name'])
                bbox.append(ann['bbox'])
                img_h.append(item['height'])
                img_w.append(item['width'])
                disease.append(diseases_dict[int(ann['category_id_3'])])

    return pd.DataFrame({'File_Name':f_name,
                    'Bbox':bbox,
                    'Height':img_h,
                    'Width': img_w,
                    'Disease_Name':disease})

df = create_dentex_df(diagnosis_map)
df

,File_Name,Bbox,Height,Width,Disease_Name
0,train_673.png,"[542.0, 698.0, 220.0, 271.0]",1316,2744,impacted
1,train_673.png,"[1952.0, 693.0, 177.0, 270.0]",1316,2744,impacted
2,train_673.png,"[675.0, 708.0, 243.0, 300.0]",1316,2744,caries
3,train_673.png,"[1463.0, 725.0, 98.0, 425.0]",1316,2744,caries
4,train_673.png,"[1536.0, 753.0, 103.0, 381.0]",1316,2744,caries
...,...,...,...,...,...
3524,train_370.png,"[1851.2857142857142, 474.2857142857142, 117.14...",1316,2948,caries
3525,train_370.png,"[1959.0, 479.9999999999999, 127.0, 274.2857142...",1316,2948,caries
3526,train_370.png,"[2024.9999999999998, 463.0, 147.00000000000023...",1316,2948,deep_caries
3527,train_370.png,"[1921.7142857142856, 749.0, 221.28571428571445...",1316,2948,caries


In [5]:
df.to_csv(r'..\Data\Processed\data (before spliting).csv')

In [7]:
df['Disease_Name'].value_counts()

Disease_Name
caries         2189
impacted        604
deep_caries     578
periapical      158
Name: count, dtype: int64